# ShadowFox Data Science Internship — Beginner Level Task
## Matplotlib & Seaborn — Complete Visualization Guide (Jupyter Notebook)

This notebook is the companion Jupyter Notebook to the report
**ShadowFox_Matplotlib_Seaborn_Report.docx**. It contains fully executable
code for:

1. Library overview / installation checks
2. Sample dataset construction (`tips`, `iris`, `flights`)
3. Eleven Matplotlib chart types
4. Eleven Seaborn chart types
5. A short, code-based comparison summary

Run all cells top-to-bottom (`Kernel > Restart & Run All`) to reproduce
every chart in the report.


## 1. Installation Check
Confirms Matplotlib and Seaborn are installed and prints their versions.

In [ ]:
import matplotlib
import seaborn as sns

print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)

## 2. Imports and Global Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde

OUTPUT_DIR = "output_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)
sns.set_theme(style="whitegrid")
%matplotlib inline

## 3. Building the Sample Datasets

The official Seaborn tutorial normally loads sample data with
`sns.load_dataset('tips')`, `sns.load_dataset('iris')`, and
`sns.load_dataset('flights')`. Those helpers require internet access to
download CSV files from GitHub. The cell below builds statistically similar
datasets **locally** so the notebook runs fully offline.

> If you have internet access, you may replace this cell with:
> `tips = sns.load_dataset('tips')`, `iris = sns.load_dataset('iris')`,
> `flights = sns.load_dataset('flights')`.


In [ ]:
n = 244
days = np.random.choice(["Thur", "Fri", "Sat", "Sun"], size=n, p=[0.25, 0.08, 0.35, 0.32])
sex = np.random.choice(["Male", "Female"], size=n)
smoker = np.random.choice(["Yes", "No"], size=n, p=[0.38, 0.62])
time_ = np.where(np.isin(days, ["Sat", "Sun"]), "Dinner",
                  np.random.choice(["Lunch", "Dinner"], size=n))
total_bill = np.round(np.random.gamma(shape=5, scale=4, size=n) + 3, 2)
tip = np.round(total_bill * np.random.normal(0.16, 0.05, n).clip(0.05, 0.35), 2)
size_ = np.random.choice([1, 2, 2, 3, 4, 4, 5, 6], size=n)

tips = pd.DataFrame({
    "total_bill": total_bill, "tip": tip, "sex": sex, "smoker": smoker,
    "day": pd.Categorical(days, categories=["Thur", "Fri", "Sat", "Sun"], ordered=True),
    "time": time_, "size": size_
})
tips.head()

In [ ]:
species = np.repeat(["setosa", "versicolor", "virginica"], 50)
sepal_length = np.concatenate([np.random.normal(5.0, 0.35, 50),
                                np.random.normal(5.9, 0.5, 50),
                                np.random.normal(6.6, 0.6, 50)])
sepal_width = np.concatenate([np.random.normal(3.4, 0.38, 50),
                               np.random.normal(2.8, 0.3, 50),
                               np.random.normal(3.0, 0.3, 50)])
petal_length = np.concatenate([np.random.normal(1.5, 0.17, 50),
                                np.random.normal(4.3, 0.47, 50),
                                np.random.normal(5.6, 0.55, 50)])
iris = pd.DataFrame({
    "sepal_length": sepal_length, "sepal_width": sepal_width,
    "petal_length": petal_length, "species": species
})
iris.head()

In [ ]:
years = list(range(1949, 1961))
flights = pd.DataFrame({
    "year": np.repeat(years, 12),
    "month": np.tile(range(1, 13), len(years)),
    "passengers": (np.linspace(112, 622, len(years) * 12) +
                    np.random.normal(0, 15, len(years) * 12)).round().astype(int)
})
flights.head()

---
# Part A — Matplotlib Chart Gallery (11 chart types)

### A.1 Line Plot
A line plot connects sequential data points, ideal for visualising trends over a continuous variable such as time.

In [ ]:
x = np.linspace(0, 10, 100)
y = np.sin(x) + np.random.normal(0, 0.1, 100)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y, color="#1f77b4", linewidth=2, label="sin(x) + noise")
ax.set_title("Matplotlib Line Plot")
ax.set_xlabel("X values")
ax.set_ylabel("Y values")
ax.legend()
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_line.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.2 Scatter Plot
A scatter plot shows the relationship between two numeric variables; colour here encodes party size.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(tips["total_bill"], tips["tip"], c=tips["size"], cmap="viridis", alpha=0.8)
ax.set_title("Matplotlib Scatter Plot")
ax.set_xlabel("Total Bill ($)")
ax.set_ylabel("Tip ($)")
fig.colorbar(sc, label="Party Size")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.3 Bar Chart
A bar chart compares an aggregated numeric value (average total bill) across categories (days).

In [ ]:
day_avg = tips.groupby("day", observed=True)["total_bill"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(day_avg.index, day_avg.values, color="#4c72b0")
ax.set_title("Matplotlib Bar Chart")
ax.set_xlabel("Day")
ax.set_ylabel("Average Total Bill ($)")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_bar.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.4 Histogram
A histogram shows the distribution of a single numeric variable (total bill amounts).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(tips["total_bill"], bins=20, color="#55a868", edgecolor="black")
ax.set_title("Matplotlib Histogram")
ax.set_xlabel("Total Bill ($)")
ax.set_ylabel("Frequency")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_histogram.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.5 Pie Chart
A pie chart shows the proportional share of restaurant visits recorded on each day.

In [ ]:
day_counts = tips["day"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(day_counts.values, labels=day_counts.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("pastel"))
ax.set_title("Matplotlib Pie Chart")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_pie.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.6 Box Plot
A box plot summarises the distribution of total bill per day using quartiles and outliers.

In [ ]:
categories = list(tips["day"].cat.categories)
data_by_day = [tips.loc[tips["day"] == d, "total_bill"] for d in categories]

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot(data_by_day, tick_labels=categories)
ax.set_title("Matplotlib Box Plot")
ax.set_xlabel("Day")
ax.set_ylabel("Total Bill ($)")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_box.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.7 Violin Plot
A violin plot combines a box plot with a density estimate to show distribution shape per day.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.violinplot(data_by_day, showmedians=True)
ax.set_xticks(range(1, len(categories) + 1))
ax.set_xticklabels(categories)
ax.set_title("Matplotlib Violin Plot")
ax.set_xlabel("Day")
ax.set_ylabel("Total Bill ($)")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_violin.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.8 Heatmap
A heatmap visualises the correlation matrix between total_bill, tip and size.

In [ ]:
corr = tips[["total_bill", "tip", "size"]].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Matplotlib Heatmap (Correlation Matrix)")
fig.colorbar(im)
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.9 Count Plot (via bar())
Matplotlib has no native countplot(); it is built manually using value_counts() + bar().

In [ ]:
counts = tips["day"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index.astype(str), counts.values, color="#c44e52")
ax.set_title("Matplotlib 'Count Plot' (built via bar())")
ax.set_xlabel("Day")
ax.set_ylabel("Count")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_countplot.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.10 Pair Plot (manual grid)
Matplotlib has no native pairplot(); it is assembled manually with a grid of subplots().

In [ ]:
cols = ["sepal_length", "sepal_width", "petal_length"]
species_colors = {"setosa": "#4c72b0", "versicolor": "#55a868", "virginica": "#c44e52"}

fig, axes = plt.subplots(len(cols), len(cols), figsize=(7, 7))
for i, ci in enumerate(cols):
    for j, cj in enumerate(cols):
        ax = axes[i, j]
        if i == j:
            for sp, color in species_colors.items():
                ax.hist(iris.loc[iris["species"] == sp, ci], color=color, alpha=0.5, bins=10)
        else:
            for sp, color in species_colors.items():
                subset = iris[iris["species"] == sp]
                ax.scatter(subset[cj], subset[ci], color=color, s=10, alpha=0.7)
        if i == len(cols) - 1:
            ax.set_xlabel(cj)
        if j == 0:
            ax.set_ylabel(ci)
fig.suptitle("Matplotlib 'Pair Plot' (manually constructed grid)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_pairplot.png"), dpi=150, bbox_inches="tight")
plt.show()

### A.11 KDE Plot (via SciPy)
Matplotlib has no native KDE function; scipy.stats.gaussian_kde is used to compute the curve.

In [ ]:
values = tips["total_bill"].values
kde = gaussian_kde(values)
xs = np.linspace(values.min(), values.max(), 200)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, kde(xs), color="#8172b2", linewidth=2)
ax.fill_between(xs, kde(xs), alpha=0.3, color="#8172b2")
ax.set_title("Matplotlib KDE Plot (via scipy.stats.gaussian_kde)")
ax.set_xlabel("Total Bill ($)")
ax.set_ylabel("Density")
fig.savefig(os.path.join(OUTPUT_DIR, "mpl_kde.png"), dpi=150, bbox_inches="tight")
plt.show()

---
# Part B — Seaborn Chart Gallery (11 chart types)

### B.1 Line Plot
sns.lineplot() draws a trend line and can automatically aggregate repeated x-values.

In [ ]:
avg_by_year = flights.groupby("year")["passengers"].mean().reset_index()

fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(data=avg_by_year, x="year", y="passengers", marker="o", ax=ax)
ax.set_title("Seaborn Line Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_line.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.2 Scatter Plot
sns.scatterplot() maps extra DataFrame columns directly to hue and marker style.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time", style="smoker", ax=ax)
ax.set_title("Seaborn Scatter Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.3 Bar Chart
sns.barplot() plots an aggregate statistic with automatic error bars, split further by hue.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=tips, x="day", y="total_bill", hue="sex", errorbar="sd", ax=ax)
ax.set_title("Seaborn Bar Chart")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_bar.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.4 Histogram
sns.histplot() draws a histogram with an optional KDE overlay, split by a hue variable.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(data=tips, x="total_bill", hue="time", kde=True, ax=ax)
ax.set_title("Seaborn Histogram")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_histogram.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.5 Pie Chart (via Matplotlib + Seaborn palette)
Seaborn has no native pie chart; Matplotlib's pie() is styled with a Seaborn colour palette.

In [ ]:
day_counts = tips["day"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(day_counts.values, labels=day_counts.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("Set2"))
ax.set_title("Pie Chart styled with a Seaborn Palette")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_pie.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.6 Box Plot
sns.boxplot() supports a hue dimension for an additional categorical split.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=tips, x="day", y="total_bill", hue="smoker", ax=ax)
ax.set_title("Seaborn Box Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_box.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.7 Violin Plot
sns.violinplot() supports split=True to compare two hue groups within one violin shape.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.violinplot(data=tips, x="day", y="total_bill", hue="sex", split=True, ax=ax)
ax.set_title("Seaborn Violin Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_violin.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.8 Heatmap
sns.heatmap() adds built-in cell annotations and colour-bar placement.

In [ ]:
corr = tips[["total_bill", "tip", "size"]].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Seaborn Heatmap (Correlation Matrix)")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.9 Count Plot
sns.countplot() counts and plots category frequencies directly from raw data.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=tips, x="day", hue="sex", ax=ax)
ax.set_title("Seaborn Count Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_countplot.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.10 Pair Plot
sns.pairplot() automatically builds a full pairwise grid for every numeric column.

In [ ]:
g = sns.pairplot(iris, hue="species", vars=["sepal_length", "sepal_width", "petal_length"],
                  diag_kind="hist", height=2.2)
g.fig.suptitle("Seaborn Pair Plot", y=1.02)
g.savefig(os.path.join(OUTPUT_DIR, "sns_pairplot.png"), dpi=150, bbox_inches="tight")
plt.show()

### B.11 KDE Plot
sns.kdeplot() is a first-class KDE function supporting hue grouping and fill shading.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.kdeplot(data=tips, x="total_bill", hue="time", fill=True, ax=ax)
ax.set_title("Seaborn KDE Plot")
fig.savefig(os.path.join(OUTPUT_DIR, "sns_kde.png"), dpi=150, bbox_inches="tight")
plt.show()

---
# Part C — Comparison Summary

| Criteria | Matplotlib | Seaborn |
|---|---|---|
| Ease of Use | Lower-level, more code | Higher-level, concise |
| Customization | Extremely high | Good, but may need Matplotlib for deep tweaks |
| Performance | Fast for basic charts | Slightly slower on very large datasets |
| Interactivity | Limited natively | Same limitation (renders through Matplotlib) |
| Large Datasets | Efficient with NumPy arrays | Can slow down with built-in statistical estimation |
| Learning Curve | Steeper | Gentler, good defaults |
| Best Use Cases | Custom, publication-quality figures | Fast statistical EDA |

## Conclusion
Matplotlib and Seaborn are complementary: Seaborn is built on Matplotlib and
returns native Matplotlib `Figure`/`Axes` objects, so the most effective
workflow is to build charts quickly with Seaborn and fine-tune them with
Matplotlib when extra control is required.
